# Naari AI — embed the KB and build the Qdrant collection

Phase 1, Sana's task: embed all 2,000 Sindhi KB questions with bge-m3 (dense + sparse) and load them into a Qdrant Cloud collection.

Runs on Colab because bge-m3 needs more RAM than a constrained local machine has free. **Runtime → Change runtime type → T4 GPU** before running (Runtime menu, top left) — it'll still work on CPU, just slower.

You'll need your Qdrant Cloud URL and API key from cloud.qdrant.io (Cluster → API Keys). The notebook prompts for them with `getpass` so they're never saved into this file.

In [ ]:
!pip install -q qdrant-client FlagEmbedding

In [ ]:
# Clone the repo so we use the *real* normalize_sd() and schema validator,
# not a copy-pasted version that can drift out of sync with retrieval/.
!rm -rf naari-ai
!git clone --branch sana/test-protection --depth 1 https://github.com/sana200420/naari-ai.git
%cd naari-ai

import sys
sys.path.insert(0, ".")

from retrieval.normalize import normalize_sd
from retrieval.schema import validate_kb_schema, DEFAULT_KB_PATH

print("cloned + imported OK")

In [ ]:
import csv

problems = validate_kb_schema()
assert problems == [], f"KB failed schema validation: {problems[:5]}"

with open(DEFAULT_KB_PATH, encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f))

print(f"{len(rows)} rows loaded and schema-valid")
print(rows[0])

## Load bge-m3 and embed all 2,000 questions (dense + sparse)

In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print("model loaded")

In [ ]:
# We embed the *question* text, not the answer — retrieval matches a user's
# query against KB questions, then the payload carries the answer to return.
# This is also the anchor Lever 2's colloquial variants will point at via
# the same answer_id, once that lands.
normalised_questions = [normalize_sd(row["question"]) for row in rows]

assert len(normalised_questions) == len(rows) == 2000
print(normalised_questions[0])

In [ ]:
BATCH = 64
dense_vecs = []
sparse_vecs = []

for i in range(0, len(normalised_questions), BATCH):
    batch = normalised_questions[i:i + BATCH]
    out = model.encode(batch, return_dense=True, return_sparse=True, return_colbert_vecs=False)
    dense_vecs.extend(out["dense_vecs"])
    sparse_vecs.extend(out["lexical_weights"])
    print(f"embedded {min(i + BATCH, len(normalised_questions))}/{len(normalised_questions)}", end="\r")

print(f"\ndone. dense dim: {len(dense_vecs[0])}")

## Connect to Qdrant and create the collection

In [ ]:
from getpass import getpass

QDRANT_URL = getpass("Qdrant cluster URL: ")
QDRANT_API_KEY = getpass("Qdrant API key: ")

In [ ]:
from qdrant_client import QdrantClient, models

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

COLLECTION = "naari_ai_kb"

client.recreate_collection(
    collection_name=COLLECTION,
    vectors_config={
        "dense": models.VectorParams(size=len(dense_vecs[0]), distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(),
    },
)
print(f"collection '{COLLECTION}' created")

In [ ]:
def to_sparse_vector(lexical_weights: dict) -> models.SparseVector:
    indices = [int(k) for k in lexical_weights.keys()]
    values = [float(v) for v in lexical_weights.values()]
    return models.SparseVector(indices=indices, values=values)

points = []
for row, dense, sparse in zip(rows, dense_vecs, sparse_vecs):
    points.append(
        models.PointStruct(
            id=int(row["id"]),
            vector={
                "dense": dense.tolist(),
                "sparse": to_sparse_vector(sparse),
            },
            payload={
                "answer_id": int(row["id"]),
                "category": row["category"],
                "sub_category": row["sub_category"],
                "question": row["question"],
                "answer": row["answer"],
                "source": row["source"],
                "review_tier": row["review_tier"],
                "lang": "sd",
            },
        )
    )

UPSERT_BATCH = 128
for i in range(0, len(points), UPSERT_BATCH):
    client.upsert(collection_name=COLLECTION, points=points[i:i + UPSERT_BATCH])
    print(f"upserted {min(i + UPSERT_BATCH, len(points))}/{len(points)}", end="\r")

print()
info = client.get_collection(COLLECTION)
print(f"collection now holds {info.points_count} points")
assert info.points_count == 2000

## Sanity check — a real query against the live collection

In [ ]:
test_query = normalize_sd("مهيني وارا ڏينهن دير سان اچن ٿا")  # colloquial, not FAQ phrasing
q_out = model.encode([test_query], return_dense=True, return_sparse=False, return_colbert_vecs=False)
q_vec = q_out["dense_vecs"][0].tolist()

hits = client.query_points(
    collection_name=COLLECTION,
    query=q_vec,
    using="dense",
    limit=5,
    with_payload=True,
).points

for h in hits:
    print(f"{h.score:.3f}  id={h.payload['answer_id']}  {h.payload['question']}")

## Next steps

- If the top hits above are plausibly about late/irregular periods, dense-only retrieval is basically working — record what you see in `docs/status.md`.
- Sparse-only and fused (RRF) queries + the ablation table are the next Playbooks task (Lever 3), once this dense baseline is confirmed working.
- The dense-only Recall@1/@5/@20 numbers against a *trustworthy* gold set are still blocked on the gold_eval.csv independence issue — don't treat this notebook's single manual query as a real recall measurement.